# Opening Range Breakout (ORBO) on SPY
## Strategy Brief -- The Opening Range Breakout (ORBO) strategy aims to capture price movements following the initial volatility of the trading day. The strategy identifies a breakout from the high or low of the first 30 minutes of trading. If the price breaks above the high, it signals a buy; if it breaks below the low, it signals a sell. The goal is to capitalize on the momentum following the breakout. Results can vary based on market conditions, but the strategy often benefits from strong directional moves.
## References
- https://www.merriam-webster.com/dictionary/opening

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for the Opening Range Breakout strategy. These include the time frame for the opening range and the trading instrument, SPY.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'
ORBO_PERIOD = '30min'

## PHASE 2 - Data Exploration
We will download historical data for SPY using yfinance. The data will be used to calculate the opening range and plot it alongside the price data to visualize potential breakouts.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download historical data for SPY
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE, interval='1d')

# Calculate the opening range high and low for the first 30 minutes of each day
data['Date'] = data.index.date
daily_data = data.resample('1D').agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'})

# Plot the closing price
data['Close'].plot(title='SPY Closing Price', figsize=(14, 7))
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we create signals based on the opening range breakout logic. We will generate a signal series indicating buy or sell signals based on the breakout of the opening range.

In [ ]:
def generate_signals(data):
    signals = pd.DataFrame(index=data.index)
    signals['signal'] = 0
    signals['high'] = data['High'].shift(1)
    signals['low'] = data['Low'].shift(1)
    signals.loc[data['Close'] > signals['high'], 'signal'] = 1
    signals.loc[data['Close'] < signals['low'], 'signal'] = -1
    return signals

signals = generate_signals(daily_data)
print(signals.head())

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by shifting the signals to avoid look-ahead bias and calculate daily returns based on the generated signals. The equity curve will be plotted to visualize the performance of the strategy.

In [ ]:
positions = signals['signal'].shift(1)
returns = daily_data['Close'].pct_change()
strategy_returns = positions * returns

# Calculate the equity curve
equity_curve = (1 + strategy_returns).cumprod()

equity_curve.plot(title='Equity Curve', figsize=(14, 7))
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the ORBO strategy by calculating key performance metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. A comparison table will be provided against a buy-and-hold strategy.

In [ ]:
import numpy as np

def calculate_performance(equity_curve):
    total_return = equity_curve.iloc[-1] - 1
    n_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (equity_curve.iloc[-1]) ** (1 / n_years) - 1
    
    # Sharpe ratio
    sharpe_ratio = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)
    
    # Sortino ratio
    downside_returns = strategy_returns.copy()
    downside_returns[downside_returns > 0] = 0
    sortino_ratio = np.mean(strategy_returns) / np.std(downside_returns) * np.sqrt(252)
    
    # Maximum drawdown
    rolling_max = equity_curve.cummax()
    drawdown = equity_curve / rolling_max - 1
    max_drawdown = drawdown.min()
    
    # Calmar ratio
    calmar_ratio = cagr / abs(max_drawdown)
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance(equity_curve)

# Buy and hold performance
buy_and_hold_returns = daily_data['Close'].pct_change().cumsum()
buy_and_hold_cagr = (1 + buy_and_hold_returns.iloc[-1]) ** (1 / n_years) - 1

comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'ORBO Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy and Hold': [buy_and_hold_cagr, None, None, None, None]
})

print(comparison_table)

## PHASE 6 - Deploy & Monitor
In the final phase, we will create a function to download the last 60 days of data, compute today's signal, and print the position for the next trading day. This will help in monitoring the strategy in real-time.

In [ ]:
def get_latest_signal():
    recent_data = yf.download(SYMBOL, period='60d', interval='1d')
    recent_daily_data = recent_data.resample('1D').agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'})
    recent_signals = generate_signals(recent_daily_data)
    latest_signal = recent_signals['signal'].iloc[-1]
    return 'Buy' if latest_signal == 1 else 'Sell' if latest_signal == -1 else 'Hold'

print('Latest Position:', get_latest_signal())